# PaperText - Phase 3C: Few-Shot Prompt

**Optimization:** Phase 3C - Few-Shot prompting

**Dataset:** PaperText (Scientific Papers - Text Q&A)

**Model:** NVIDIA Nemotron-3 Ultra 550B (via Together AI)

**Metric:** Span F1

**Documents:** 2 example PDFs

**Q&A Count:** 13 pairs

**What changed:**
- ✅ Few-Shot prompt (2-3 domain examples)
- ✅ Keep all Phase 2 parameters (TOP_K=10, CHUNK_SIZE=3000)

**Baseline (Phase 2):**
- Empty rate: 7.7% (1/13 questions)

**Target:**
- Empty rate: <7.7% (any improvement)

**Expected runtime:** 6-9 minutes
**Expected cost:** 1.2x tokens

## Setup and Imports

In [1]:
import sys
import os

# Navigate to project root
project_root = os.path.abspath('../../../../..')
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


In [2]:
import pandas as pd
import re
import chromadb
import PyPDF2
import time
import importlib.util
from datetime import datetime
from together import Together
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb.utils.embedding_functions as embedding_functions
from uda.utils import preprocess
from uda.utils.prompts import get_prompt
from uda.eval.my_eval import eval_main

print("✓ All imports successful")

✓ All imports successful


/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configuration

In [3]:
# Load API config
_spec = importlib.util.spec_from_file_location(
    "access_config",
    os.path.join(os.getcwd(), "uda", "utils", "access_config.py")
)
access_config = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(access_config)

print(f"Model: {access_config.TOGETHER_MODEL}")
print(f"API Key: {access_config.TOGETHER_API_KEY[:20]}...")

Model: nvidia/nemotron-3-ultra-550b-a55b
API Key: tgp_v1_9OcdTuqoXTB0_...


In [4]:
# Experiment Parameters
DATASET_NAME = "paper"
CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300
TOP_K = 10
TEMPERATURE = 0.1
MAX_TOKENS = 512

# Prompt type
PROMPT_TYPE = "fewshot"

# Output settings
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = "./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/papertext_fewshot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Top-K: {TOP_K}")
print(f"Prompt type: {PROMPT_TYPE}")
print(f"Output dir: {OUTPUT_DIR}")

Dataset: paper
Chunk size: 3000
Top-K: 10
Prompt type: fewshot
Output dir: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/papertext_fewshot


## Initialize Models

In [5]:
# Together AI client
together_client = Together(api_key=access_config.TOGETHER_API_KEY)
print("✓ Together AI client initialized")

# Embedding model
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
print("✓ Embedding model loaded: all-MiniLM-L6-v2")

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
print("✓ Text splitter initialized")

# Load prompt function
prompt_fn = get_prompt(PROMPT_TYPE)
print(f"✓ Prompt function loaded: {PROMPT_TYPE}")

/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Together AI client initialized
✓ Embedding model loaded: all-MiniLM-L6-v2
✓ Text splitter initialized
✓ Prompt function loaded: fewshot


## Helper Functions

In [6]:
def extract_pdf_text(pdf_path):
    """Extract text from PDF using PyPDF2"""
    pdf_text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file, strict=False)
        for page_num in range(len(reader.pages)):
            pdf_text += reader.pages[page_num].extract_text()
    return pdf_text

def sanitize_collection_name(doc_name, dataset_name):
    """
    Sanitize document name for ChromaDB collection.
    
    ChromaDB requires: 3-512 chars from [a-zA-Z0-9._-], 
    starting and ending with alphanumeric.
    """
    # Replace invalid chars with underscore
    safe_name = re.sub(r'[^a-zA-Z0-9._-]', '_', doc_name)
    # Remove consecutive underscores
    safe_name = re.sub(r'_+', '_', safe_name)
    # Remove leading/trailing underscores
    safe_name = safe_name.strip('_')
    # Prepend dataset name
    collection_name = f"{dataset_name}_{safe_name}"
    return collection_name

def build_index(text_chunks, collection_name="temp_collection"):
    """Build vector index"""
    chroma_client = chromadb.Client()

    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass

    collection = chroma_client.create_collection(
        collection_name,
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"}
    )

    id_list = [str(i) for i in range(len(text_chunks))]
    collection.add(documents=text_chunks, ids=id_list)

    return collection

def answer_question(collection, question):
    """
    Retrieve context and generate answer.

    CHANGED: Uses PROMPT_TYPE prompt
    """
    # Retrieve
    fetch_res = collection.query(query_texts=[question], n_results=TOP_K)
    context = "\n".join(fetch_res["documents"][0])

    # Build prompt using prompts module
    prompt_text = prompt_fn(context=context, question=question)

    # Convert to message format
    messages = [
        {"role": "user", "content": prompt_text}
    ]

    # Generate
    response = together_client.chat.completions.create(
        model=access_config.TOGETHER_MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )

    return response.choices[0].message.content

print("✓ Helper functions defined")


✓ Helper functions defined


## Load Q&A Data

In [7]:
# Load Q&A
csv_file = "./dataset/qa/paper_text_qa.csv"
df = pd.read_csv(csv_file, sep="|", na_filter=False, dtype={"doc_name": str})
qas_dict_all = preprocess.qa_df_to_dict(DATASET_NAME, df)

# Filter to documents with available PDFs - PHASE 2 DOCUMENT LIST
AVAILABLE_DOCS = [
    "1705.07830",
    "1801.05147",
    "1809.01202",
    "1810.08699",
    "1909.00754",
    "1912.01214",
    "2001.03131"
]

qas_dict = {doc: qas for doc, qas in qas_dict_all.items() if doc in AVAILABLE_DOCS}

print(f"Total documents in CSV: {len(qas_dict_all)}")
print(f"Available PDFs: {len(AVAILABLE_DOCS)}")
print(f"\nFiltered to documents with PDFs:\n")

total_qa = 0
for doc in AVAILABLE_DOCS:
    if doc in qas_dict:
        count = len(qas_dict[doc])
        total_qa += count
        print(f"  {doc}: {count} Q&A pairs")

print(f"\nTotal Q&A to process: {total_qa}")


Total documents in CSV: 1087
Available PDFs: 7

Filtered to documents with PDFs:

  1705.07830: 1 Q&A pairs
  1801.05147: 1 Q&A pairs
  1809.01202: 1 Q&A pairs
  1810.08699: 3 Q&A pairs
  1909.00754: 2 Q&A pairs
  1912.01214: 3 Q&A pairs
  2001.03131: 2 Q&A pairs

Total Q&A to process: 13


## Main Processing Loop

**Expected runtime:** 6-9 minutes

In [8]:
all_results = []

for doc_name, doc_qas in qas_dict.items():
    print(f"\n{'='*80}")
    print(f"Processing: {doc_name}")
    print(f"{'='*80}")

    # Get PDF path
    pdf_path = preprocess.get_example_pdf_path(DATASET_NAME, doc_name)
    if not pdf_path:
        print(f"❌ PDF not found - skipping")
        continue

    print(f"PDF: {pdf_path}")

    # Extract and chunk
    print("Extracting text...")
    pdf_text = extract_pdf_text(pdf_path)
    text_chunks = text_splitter.split_text(pdf_text)
    print(f"Created {len(text_chunks)} chunks")

    # Build index
    print("Building vector index...")
    collection_name = sanitize_collection_name(doc_name, DATASET_NAME)
    collection = build_index(text_chunks, collection_name=collection_name)
    print("✓ Index built")

    # Process questions
    print(f"\nAnswering {len(doc_qas)} questions...")

    for idx, qa in enumerate(doc_qas, 1):
        question = qa["question"]
        print(f"\n[{idx}/{len(doc_qas)}] {question[:70]}...")

        try:
            answer = answer_question(collection, question)
            print(f"   Answer: {answer[:80]}...")

            all_results.append({
                "question": question,
                "response": answer,
                "doc": doc_name,
                "q_uid": qa["q_uid"],
                "answers": qa["answers"],
                "dataset": DATASET_NAME,
                "prompt_type": PROMPT_TYPE,
            })

            time.sleep(0.5)

        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue

    print(f"\n✓ Completed {doc_name}")

print(f"\n{'='*80}")
print(f"ALL DOCUMENTS PROCESSED")
print(f"{'='*80}")
print(f"Total Q&A processed: {len(all_results)}")



Processing: 1912.01214
PDF: dataset/src_doc_files_example/paper_docs/1912.01214.pdf
Extracting text...
Created 14 chunks
Building vector index...
✓ Index built

Answering 3 questions...

[1/3] which multilingual approaches do they compare with?...
   Answer: They compare with Multilingual NMT (MNMT) by Johnson et al. (2016) and MNMT Agre...

[2/3] what are the pivot-based baselines?...
   Answer: Based on the provided context, the pivot-based baselines used in the experiments...

[3/3] which datasets did they experiment with?...
   Answer: The experiments were conducted on two public datasets: Europarl and MultiUN....

✓ Completed 1912.01214

Processing: 1810.08699
PDF: dataset/src_doc_files_example/paper_docs/1810.08699.pdf
Extracting text...
Created 12 chunks
Building vector index...
✓ Index built

Answering 3 questions...

[1/3] what ner models were evaluated?...
   Answer: Based on the context, three named entity recognition models were evaluated:

1. ...

[2/3] what is the source

## Diagnostic: Check Empty Responses

In [9]:
if all_results:
    results_df = pd.DataFrame(all_results)

    # Count empty responses
    results_df['is_empty'] = results_df['response'].fillna('').str.strip() == ''
    empty_count = results_df['is_empty'].sum()
    total_count = len(results_df)

    print(f"\n{'='*80}")
    print(f"DIAGNOSTIC: Empty Response Analysis")
    print(f"{'='*80}")
    print(f"Total Q&A processed: {total_count}")
    print(f"Empty responses: {empty_count} ({empty_count/total_count*100:.1f}%)")
    print(f"Answered: {total_count - empty_count} ({(total_count-empty_count)/total_count*100:.1f}%)")

    # Comparison with Phase 2
    phase2_empty = 1
    phase2_total = 13
    phase2_empty_pct = 7.7

    improvement = phase2_empty - empty_count
    improvement_pct = phase2_empty_pct - (empty_count/total_count*100)

    print(f"\n{'='*80}")
    print(f"COMPARISON WITH PHASE 2 BASELINE")
    print(f"{'='*80}")
    print(f"Phase 2 (Baseline): {phase2_empty}/{phase2_total} empty ({phase2_empty_pct:.1f}%)")
    print(f"Phase 3C (Few-Shot): {empty_count}/{total_count} empty ({empty_count/total_count*100:.1f}%)")
    print(f"\nImprovement: {improvement:+d} questions ({improvement_pct:+.1f} percentage points)")

    if improvement > 0:
        print(f"✅ SUCCESS: Few-Shot reduced empty responses!")
    elif improvement == 0:
        print(f"⚠️  NEUTRAL: No change")
    else:
        print(f"❌ REGRESSION: Empty responses increased")

    if empty_count > 0:
        print(f"\nEmpty responses by document:")
        for doc in results_df['doc'].unique():
            doc_df = results_df[results_df['doc'] == doc]
            doc_empty = doc_df['is_empty'].sum()
            doc_total = len(doc_df)
            print(f"  {doc}: {doc_empty}/{doc_total} empty ({doc_empty/doc_total*100:.1f}%)")
else:
    print("❌ No results to analyze")


DIAGNOSTIC: Empty Response Analysis
Total Q&A processed: 13
Empty responses: 0 (0.0%)
Answered: 13 (100.0%)

COMPARISON WITH PHASE 2 BASELINE
Phase 2 (Baseline): 1/13 empty (7.7%)
Phase 3C (Few-Shot): 0/13 empty (0.0%)

Improvement: +1 questions (+7.7 percentage points)
✅ SUCCESS: Few-Shot reduced empty responses!


## Evaluate Results

In [10]:
if all_results:
    print(f"\nEvaluating {DATASET_NAME} results...")
    eval_main(DATASET_NAME, all_results)
else:
    print("❌ No results to evaluate")


Evaluating paper results...
{'Answer F1': 0.26153388883425, 'Missing predictions': 0}


## Save Results

In [11]:
if all_results:
    results_df = pd.DataFrame(all_results)
    output_file = os.path.join(OUTPUT_DIR, f"papertext_fewshot_{TIMESTAMP}.csv")
    results_df.to_csv(output_file, index=False)

    print(f"\n✓ Results saved to: {output_file}")
    print(f"Total Q&A: {len(results_df)}")

    print("\nResults by document:")
    for doc in results_df['doc'].unique():
        count = len(results_df[results_df['doc'] == doc])
        print(f"  {doc}: {count} questions")
else:
    print("❌ No results to save")


✓ Results saved to: ./experiments/nemotron-3-ultra-550b/3_advanced_optimization/3_prompts/results/papertext_fewshot/papertext_fewshot_20260630_124013.csv
Total Q&A: 13

Results by document:
  1912.01214: 3 questions
  1810.08699: 3 questions
  1801.05147: 1 questions
  1809.01202: 1 questions
  2001.03131: 2 questions
  1909.00754: 2 questions
  1705.07830: 1 questions


## Final Summary

In [12]:
if all_results:
    results_df = pd.DataFrame(all_results)

    empty_count = results_df['response'].fillna('').str.strip().eq('').sum()
    answered_count = len(results_df) - empty_count

    phase2_empty = 1
    improvement = phase2_empty - empty_count

    print(f"\n{'='*80}")
    print(f"FINAL SUMMARY - FEW-SHOT (PaperText)")
    print(f"{'='*80}")
    print(f"Dataset: PaperText ({len(results_df)} Q&A)")
    print(f"Prompt type: {PROMPT_TYPE}")
    print(f"\nResults:")
    print(f"  Answered: {answered_count}/{len(results_df)} ({answered_count/len(results_df)*100:.1f}%)")
    print(f"  Empty: {empty_count}/{len(results_df)} ({empty_count/len(results_df)*100:.1f}%)")
    print(f"\nVs Phase 2 Baseline:")
    print(f"  Change: {improvement:+d} questions")
    print(f"  Cost: 1.2x tokens")

    if improvement >= 2:
        print(f"\n✅ EXCELLENT: Few-Shot significantly improved!")
    elif improvement >= 1:
        print(f"\n✅ GOOD: Few-Shot helped")
    elif improvement == 0:
        print(f"\n⚠️  NEUTRAL: No change")
    else:
        print(f"\n❌ REGRESSION: Made things worse")
else:
    print("\n❌ No results to summarize")


FINAL SUMMARY - FEW-SHOT (PaperText)
Dataset: PaperText (13 Q&A)
Prompt type: fewshot

Results:
  Answered: 13/13 (100.0%)
  Empty: 0/13 (0.0%)

Vs Phase 2 Baseline:
  Change: +1 questions
  Cost: 1.2x tokens

✅ GOOD: Few-Shot helped


---

## Done!

**Results saved to:** `./results/papertext_fewshot/`

Compare with other prompt types to find the best approach for PaperText.